<a href="https://colab.research.google.com/github/NagaRaju1991/google_colab_notebooks/blob/fsds_course/02_Embeddings_word2Vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 47.8 MB/s eta 0:00:00


### Word2Vec Example

In [ ]:
from gensim.models import Word2Vec

# 1. Sample Data: A list of tokenized sentences
sentences = [
    ["the", "king", "rules", "the", "kingdom"],
    ["the", "queen", "rules", "the", "palace"],
    ["a", "man", "is", "strong"],
    ["a", "woman", "is", "wise"]
]

# 2. Train the Model
# vector_size=100: each word becomes a list of 100 numbers
# window=5: look at 5 words to the left and right
# min_count=1: ignore words that appear fewer than 1 time
model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

# 3. Find Similarities
similar_to_king = model.wv.most_similar("king")
print(f"Words similar to 'king': {similar_to_king}")

# 4. Get the actual vector for a word
king_vector = model.wv['king']
print(f"Vector for 'king' (first 5 values): {king_vector[:5]}")

Words similar to 'king': [('a', 0.17274504899978638), ('woman', 0.1669391542673111), ('man', 0.11119871586561203), ('kingdom', 0.10941851139068604), ('the', 0.079634889960289), ('rules', 0.04130807891488075), ('palace', 0.037712957710027695), ('is', 0.00831594504415989), ('wise', -0.005896794609725475), ('queen', -0.0742427185177803)]
Vector for 'king' (first 5 values): [0.00977029 0.00816511 0.00128097 0.00509758 0.00140813]


**Glove Example**

In [ ]:
import gensim.downloader as api

# 1. Load a pre-trained GloVe model (trained on 6 Billion words from Wikipedia)
# 'glove-wiki-gigaword-100' means 100-dimensional vectors
model = api.load("glove-wiki-gigaword-100")

# 2. Check word similarity
similarity = model.similarity('king', 'queen')
print(f"Similarity between King and Queen: {similarity:.2f}")

# 3. Find the 'odd one out'
words = ['fire', 'water', 'ice', 'car']
print(f"The word that doesn't fit is: {model.doesnt_match(words)}")

# 4. The famous analogy: King - Man + Woman = ?
result = model.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)
print(f"King - Man + Woman = {result[0][0]}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Similarity between King and Queen: 0.75
The word that doesn't fit is: car
King - Man + Woman = queen


In [ ]:
result

[('queen', 0.7698540687561035)]

In [ ]:
! pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 5.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.2-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.2-py3-none-any.whl (310 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4647420 sha256=d33e78b46e84ef4375921f8b0eb400cb2229cb254aa56330ef4c95bd8b9772ed
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [ ]:
import fasttext.util

# 1. Download and load a pre-trained English model (2GB+)
# This only needs to be done once
fasttext.util.download_model('en', if_exists='ignore')
model = fasttext.load_model('cc.en.300.bin')

# 2. Get vector for a word
# Even if "AI-tastic" isn't a real word, fastText gives a vector!
word_vec = model.get_word_vector('AI-tastic')

# 3. Find nearest neighbors
neighbors = model.get_nearest_neighbors('king')
print(f"Nearest to king: {neighbors[0]}")


Nearest to king: (0.7550359964370728, 'kings')


**FastText**

In [ ]:
import torch
import torch.nn as nn

class SentimentGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(SentimentGRU, self).__init__()

        # 1. Embedding Layer: Converts word indices to dense vectors (e.g., GloVe)
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 2. GRU Layer: Processes the sequence of word vectors
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)

        # 3. Fully Connected Layer: Maps the final hidden state to a sentiment score
        self.fc = nn.Linear(hidden_dim, output_dim)

        # 4. Sigmoid: Squashes the output between 0 (Negative) and 1 (Positive)
        self.sigmoid = nn.Sigmoid()

    def forward(self, text):
        # text shape: [batch_size, seq_length]

        embedded = self.embedding(text)
        # embedded shape: [batch_size, seq_length, embedding_dim]

        _, hidden = self.gru(embedded)
        # hidden shape: [1, batch_size, hidden_dim]

        # We take the last hidden state (the summary of the sentence)
        last_hidden = hidden.squeeze(0)

        output = self.fc(last_hidden)
        return self.sigmoid(output)

# --- Simulation ---
# Setup hyper-parameters
VOCAB_SIZE = 1000  # Number of unique words in our dictionary
EMBED_DIM = 100    # Size of word vectors (e.g., GloVe 100d)
HIDDEN_DIM = 256
OUTPUT_DIM = 1     # Single value: 0 to 1

model = SentimentGRU(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM)

# Dummy review: "This movie was absolutely fantastic" (encoded as integers)
# Shape: [Batch size 1, Sequence length 5]
sample_review = torch.randint(0, VOCAB_SIZE, (1, 5))

prediction = model(sample_review)
print(f"Sentiment Probability: {prediction.item():.4f}")
print("Result:", "Positive" if prediction.item() > 0.5 else "Negative")

Sentiment Probability: 0.5323
Result: Positive


**Addtional Example**

In [ ]:
from gensim.models import Word2Vec
import re

# 1. A Larger, More Diverse Dataset
corpus = [
    "The king sat on his golden throne in the massive palace.",
    "The queen ruled the entire kingdom with wisdom and grace.",
    "A man is often defined by his physical strength and courage.",
    "A woman is recognized for her intelligence and strong leadership.",
    "The prince and princess will one day inherit the royal castle.",
    "The monarch wears a crown made of pure gold and jewels.",
    "In the laboratory, the scientist conducts data science experiments.",
    "Artificial intelligence and machine learning are the future of data.",
    "The researcher published a paper on deep learning and neural networks."
]

# 2. Preprocessing: Tokenization & Cleaning
# We lowercase and remove punctuation so 'King.' and 'king' are the same token.
processed_corpus = [re.sub(r'[^\w\s]', '', sent).lower().split() for sent in corpus]

# 3. Training the Word2Vec Model (Skip-gram)
# sg=1: Use Skip-gram (better for small datasets/rare words)
# vector_size=100: Each word is compressed into 100 numbers
# window=5: The model looks 5 words to the left and 5 to the right
model = Word2Vec(sentences=processed_corpus, vector_size=100, window=5, min_count=1, sg=1)

# 4. Performing "Word Math" (The Power of Embeddings)
# Example: King - Man + Woman = ?
result = model.wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)
print(f"Result of King - Man + Woman: {result[0][0]}")

# 5. Finding Contextual Clusters
# Even with this small data, 'data' should be closer to 'science' than to 'palace'.
sim_score = model.wv.similarity('data', 'science')
print(f"Similarity between 'data' and 'science': {sim_score:.4f}")

# 6. Extracting for PyTorch
# If you want to use these weights in a nn.Embedding layer:
word_vectors = model.wv.get_normed_vectors()
print(f"Shape of the embedding matrix: {word_vectors.shape}")
# Result: (Total unique words, 100)